# Polynomial and Fourier basis regression

**Status:** ✅ Runs end to end

**Demonstrates:** Basis functions · a fixed design matrix. The original rebuilt the basis from the data inside `predict`, so it could not generalise; the notebook now tests exactly that.

---

> Part of a portfolio of applied ML and statistics work. Every notebook
> here is executed end to end; the bug record is in `BUGFIXES.md`.


In [ ]:

import numpy as np
from matplotlib import pyplot as plt

rng = np.random.default_rng(11)


## What was broken

Three defects, and the third is the interesting one.

1. `fit()` did `i += 1` but `i` was never initialised → `NameError` on the
   first pass.
2. `un_norm` inverted the x-axis as `X *= (max + min) + min`. The inverse of
   `(x - min) / (max - min)` is `(x * (max - min)) + min`; the original did
   not undo the forward transform at all.
3. **The design matrix was rebuilt from the data inside `predict`.** Because
   the basis is data-dependent, the model was not a function of its weights
   alone, so predicting on new points was unsound - the weights had been
   fitted against one basis and applied to a different one.

The third is the real lesson: a model's basis must be fixed at fit time and
reused at predict time, or it is not a model.

## 1. Three interchangeable bases

The same linear model over three different function bases:

| `basis` | columns |
|---|---|
| `polynomial` | `1, x, x^2, ..., x^n` |
| `fourier` | `1, cos(2pi k x), sin(2pi k x)` for `k = 1..n` |
| `mixed` | both, concatenated |

In [ ]:

def build_basis(x, degree, basis="polynomial"):
    """Construct a fixed function basis evaluated at ``x``.

    BUGFIX: the original recomputed the basis from whatever ``x`` was passed
    in, inside both ``fit`` and ``predict``. That made the model data-dependent
    and unsound on new inputs. Here the basis is built explicitly and the
    caller is responsible for reusing the same ``(degree, basis)`` pair at
    predict time.

    Args:
        x: Input values of shape ``(n_samples,)``.
        degree: Highest power or harmonic number to include.
        basis: One of ``"polynomial"``, ``"fourier"`` or ``"mixed"``.

    Returns:
        numpy.ndarray: Basis matrix of shape ``(n_samples, n_columns)``.

    Raises:
        ValueError: If ``basis`` is not one of the three recognised names.

    Example:
        >>> build_basis(np.array([0.0, 0.5, 1.0]), 2).shape
        (3, 3)
    """
    x = np.asarray(x, dtype=float).reshape(-1, 1)

    polynomial = x ** np.arange(degree + 1)

    if basis == "polynomial":
        return polynomial

    harmonics = np.arange(1, degree + 1)
    fourier = np.hstack([
        np.ones_like(x),
        np.cos(2 * np.pi * harmonics.reshape(1, -1) * x),
        np.sin(2 * np.pi * harmonics.reshape(1, -1) * x),
    ])

    if basis == "fourier":
        return fourier
    if basis == "mixed":
        return np.hstack([polynomial, fourier[:, 1:]])
    raise ValueError(f"basis must be 'polynomial', 'fourier' or 'mixed', got {basis!r}")


def n_columns(degree, basis="polynomial"):
    """Return how many columns :func:`build_basis` will produce.

    Args:
        degree: Basis degree or harmonic count.
        basis: Basis name.

    Returns:
        int: Column count, used to size the weight vector.
    """
    if basis == "polynomial":
        return degree + 1
    if basis == "fourier":
        return 2 * degree + 1
    if basis == "mixed":
        return (degree + 1) + 2 * degree
    raise ValueError(f"basis must be 'polynomial', 'fourier' or 'mixed', got {basis!r}")


## 2. A scaler that is actually reversible

The forward transform is `z = (x - min) / (max - min)`. The inverse is
`x = z * (max - min) + min`. Getting that second step right is the whole fix
for the original `un_norm`.

In [ ]:

class MinMaxNormalizer:
    """Rescale a 1-D array onto ``[0, 1]`` and back again.

    BUGFIX: the original ``un_norm`` read ``X *= (max + min) + min``, which
    does not invert the forward transform. The inverse is
    ``x = z * (max - min) + min``.

    Attributes:
        minimum_ (float): Smallest value seen during :meth:`fit`.
        span_ (float): ``max - min``. ``0`` for a constant series.
    """

    def __init__(self):
        self.minimum_ = None
        self.maximum_ = None
        self.span_ = None

    def fit(self, values):
        """Record the range of ``values``.

        Args:
            values: 1-D array of any length.

        Returns:
            MinMaxNormalizer: self, for chaining.
        """
        values = np.asarray(values, dtype=float)
        self.minimum_ = float(values.min())
        self.maximum_ = float(values.max())
        self.span_ = self.maximum_ - self.minimum_
        return self

    def transform(self, values):
        """Scale values onto ``[0, 1]``.

        Args:
            values: 1-D array.

        Returns:
            numpy.ndarray: Newly allocated scaled array; the input is not
            modified.

        Raises:
            RuntimeError: If called before :meth:`fit`.
        """
        if self.minimum_ is None:
            raise RuntimeError("MinMaxNormalizer has not been fitted yet.")
        if self.span_ == 0:
            return np.zeros_like(np.asarray(values, dtype=float))
        return (np.asarray(values, dtype=float) - self.minimum_) / self.span_

    def inverse_transform(self, values):
        """Map values from ``[0, 1]`` back to the original range.

        Args:
            values: 1-D array on the unit interval.

        Returns:
            numpy.ndarray: Values on the original scale. The input is not
            modified.

        Raises:
            RuntimeError: If called before :meth:`fit`.

        Example:
            >>> n = MinMaxNormalizer().fit([2.0, 4.0, 6.0])
            >>> bool(np.allclose(n.inverse_transform(n.transform([2.0, 4.0, 6.0])),
            ...                    [2.0, 4.0, 6.0]))
            True
        """
        if self.minimum_ is None:
            raise RuntimeError("MinMaxNormalizer has not been fitted yet.")
        return np.asarray(values, dtype=float) * self.span_ + self.minimum_


# Round-trip check, the direct test of the original bug.
raw = rng.random(50) * 17 - 4
normalizer = MinMaxNormalizer().fit(raw)
round_tripped = normalizer.inverse_transform(normalizer.transform(raw))
print("max round-trip error:", float(np.max(np.abs(round_tripped - raw))))


## 3. The regressor

Weights are fitted by gradient descent on the *fixed* basis, and `predict`
reuses the same `(degree, basis)` pair.

In [ ]:

class BasisRegressor:
    """Least-squares fit over a fixed polynomial or Fourier basis.

    BUGFIX: the original incremented an uninitialised ``i`` inside ``fit``, and
    rebuilt the design matrix inside ``predict`` from the new inputs.

    Attributes:
        weights_ (numpy.ndarray): Learned coefficients, one per basis column.
        normalizer_ (MinMaxNormalizer): Fitted x-scaler.
        degree_ (int): Basis degree used at fit time.
        basis_ (str): Basis name used at fit time.
    """

    def __init__(self, degree=6, basis="polynomial", learning_rate=0.2, max_epochs=4000, tol=1e-10):
        """Configure the regressor.

        Args:
            degree: Basis degree or harmonic count.
            basis: ``"polynomial"``, ``"fourier"`` or ``"mixed"``.
            learning_rate: Gradient-descent step size.
            max_epochs: Hard cap on passes over the data.
            tol: Convergence tolerance on the squared weight change.
        """
        self.degree = degree
        self.basis = basis
        self.learning_rate = learning_rate
        self.max_epochs = max_epochs
        self.tol = tol

        self.weights_ = None
        self.normalizer_ = None
        self.degree_ = None
        self.basis_ = None

    def fit(self, x, y):
        """Fit the coefficients by gradient descent.

        BUGFIX: ``i`` is initialised to ``0`` here. The original did
        ``i += 1`` with no prior assignment, raising ``NameError``.

        Args:
            x: Inputs of shape ``(n_samples,)``.
            y: Targets of shape ``(n_samples,)``.

        Returns:
            BasisRegressor: self, for chaining.
        """
        self.normalizer_ = MinMaxNormalizer().fit(x)
        self.degree_, self.basis_ = self.degree, self.basis
        phi = build_basis(self.normalizer_.transform(x), self.degree_, self.basis_)
        target = np.asarray(y, dtype=float)

        self.weights_ = np.zeros(phi.shape[1])
        i = 0                                   # BUGFIX: was never initialised
        while i < self.max_epochs:
            i += 1
            residual = phi @ self.weights_ - target
            gradient = (phi.T @ residual) / phi.shape[0]
            previous = self.weights_.copy()
            self.weights_ -= self.learning_rate * gradient
            if np.sum((self.weights_ - previous) ** 2) < self.tol:
                break
        return self

    def predict(self, x):
        """Predict targets using the basis fixed at fit time.

        BUGFIX: the original called ``choose_model(x)`` here, which rebuilt the
        basis from the *new* x. The weights had been fitted against one basis
        and were being applied to a different one, so predictions on new points
        were meaningless.

        Args:
            x: Inputs of shape ``(n_samples,)``.

        Returns:
            numpy.ndarray: Predictions of shape ``(n_samples,)``.

        Raises:
            RuntimeError: If called before :meth:`fit`.
        """
        if self.weights_ is None:
            raise RuntimeError("BasisRegressor has not been fitted yet; call fit first.")
        phi = build_basis(self.normalizer_.transform(x), self.degree_, self.basis_)
        return phi @ self.weights_

    def score(self, x, y):
        """Return the coefficient of determination.

        Args:
            x: Inputs of shape ``(n_samples,)``.
            y: True targets of shape ``(n_samples,)``.

        Returns:
            float: :math:`R^2`, where 1.0 is a perfect fit.
        """
        y = np.asarray(y, dtype=float)
        residual = y - self.predict(x)
        total = y - y.mean()
        return float(1 - np.sum(residual ** 2) / np.sum(total ** 2))


## 4. Fit the same data three ways

In [ ]:

SAMPLE_COUNT = 600
x = np.sort(rng.random(SAMPLE_COUNT) * 6 - 3)
# A target that is partly smooth and partly oscillatory: the polynomial and the
# Fourier basis should each win on different parts of it.
y = 0.3 * x ** 2 - 0.5 * x + 0.6 * np.sin(2.2 * x) + 0.15 * rng.normal(size=SAMPLE_COUNT)

print(f"{'basis':12} {'degree':>7} {'R^2':>8}   best degree")
results = {}
for basis in ("polynomial", "fourier", "mixed"):
    scores = []
    for degree in (2, 4, 6, 8, 10):
        model = BasisRegressor(degree=degree, basis=basis).fit(x, y)
        scores.append((degree, model.score(x, y)))
    best_degree, best_score = max(scores, key=lambda pair: pair[1])
    results[basis] = (best_degree, best_score)
    print(f"{basis:12} {best_degree:7d} {best_score:8.4f}")


## 5. Does the basis actually transfer to new points?

This is the test the original design could not pass: fit on one sample of the
x range, predict on a *disjoint* range. A basis that leaks the evaluation grid
into the model degrades badly here.

In [ ]:

train_mask = x < 1.5
test_mask = ~train_mask
print(f"fit on {train_mask.sum()} points in [-3, 1.5), predict on {test_mask.sum()} in [1.5, 3)")

print(f"{'basis':12} {'train R^2':>10} {'held-out R^2':>13}")
held_out = {}
for basis in ("polynomial", "fourier", "mixed"):
    model = BasisRegressor(degree=results[basis][0], basis=basis).fit(x[train_mask], y[train_mask])
    train_r2 = model.score(x[train_mask], y[train_mask])
    test_r2 = model.score(x[test_mask], y[test_mask])
    held_out[basis] = (model, test_r2)
    print(f"{basis:12} {train_r2:10.4f} {test_r2:13.4f}")

best_basis = max(held_out, key=lambda b: held_out[b][1])
model = held_out[best_basis][0]
print(f"\nbest on held-out data: {best_basis}")

plt.figure(figsize=(7.5, 4.5))
plt.scatter(x[train_mask], y[train_mask], s=6, color="tab:gray", alpha=0.5, label="fit region")
plt.scatter(x[test_mask], y[test_mask], s=6, color="tab:orange", alpha=0.6, label="held-out")
grid = np.linspace(x.min(), x.max(), 400)
plt.plot(grid, model.predict(grid), color="tab:red", lw=1.5, label=f"{best_basis} prediction")
plt.xlabel("x")
plt.ylabel("y")
plt.title("Extrapolation to a region never seen during fitting")
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()
